# Project 1 - Content Moderation Baseline
This notebook provides a starting point for the **Human-Assisted Content Moderation** project.

We use the **Civil Comments** dataset to train a baseline classifier that identifies potentially toxic comments using TF-IDF and Logistic Regression.

You are expected to extend this baseline through your own experiments, following an **evidence-driven model development approach**. Refer to the project brief for the full project objectives and requirements.

## 1. Your Tasks

The provided notebook establishes a simple baseline for detecting potentially toxic comments. Your task is to **evaluate this baseline, identify its limitations, and investigate possible improvements**.

### 1.1 Explore the Dataset

Get familiar with the data and the classification task.

- Examine the dataset characteristics, class distribution, and representative examples.
- Consider how the target labels are defined and whether the data is suitable for the intended application.
- Identify potential issues related to data quality, coverage, or representativeness.

### 1.2 Evaluate the Baseline

Determine how well the model performs and which evaluation criteria are appropriate.

- Select and justify metrics relevant to the content moderation task.
- Evaluate overall and class-wise performance.
- Consider the consequences of different types of errors, particularly false positives and false negatives.

### 1.3 Analyze Errors and Diagnose Limitations

Investigate where and why the classifier makes mistakes.

- Examine misclassified examples and identify recurring patterns.
- Formulate possible explanations for the observed errors.
- Distinguish observations supported by the data from hypotheses that require further investigation.

### 1.4 Investigate Possible Improvements

Based on your observations and relevant literature, identify promising directions for improving the baseline.

- Research alternative approaches, representations, or model configurations.
- Explain which limitations these approaches might address.
- Where feasible, implement and evaluate an improvement against the original baseline using a consistent evaluation setup.

### 1.5 Document Your Findings

Summarize your experiments and the reasoning behind your decisions.

- What did you investigate, and why?
- What evidence supports your observations and conclusions?
- Which improvements appear promising, and what remains unresolved?

**Important:** Your decisions should be grounded in **empirical evidence or relevant research**, rather than assumptions. When proposing changes, explain the motivation and evaluate their effects whenever possible.

Use the validation set for model development and comparisons. Keep the test set reserved for final evaluation.

## 2. Starter code

We use the **Civil Comments** dataset, which contains user comments annotated with toxicity-related scores.

### 2.1 Loading and sampling the dataset

If your Jupyter notebook cannot access the Internet (e.g., on the YourAI cluster), download the dataset beforehand using the `download_dataset.py` script:

````bash
python scripts/download_dataset.py "google/civil_comments" "data/google_civil_comments"
````

In [ ]:
from datasets import load_dataset, load_from_disk

ds = load_from_disk("../../data/google_civil_comments")

ds

**Sampling**. The dataset is very big. To keep things maneagable, we can sample a subset.

In [ ]:
# Sample from each original partition
train_ds = ds["train"].train_test_split(train_size=25_000, seed=42)["train"]
val_ds = ds["validation"].train_test_split(train_size=5_000, seed=42)["train"]
test_ds = ds["test"].train_test_split(train_size=5_000, seed=42)["train"]

# Convert to Pandas
train_df = train_ds.to_pandas()
val_df = val_ds.to_pandas()
test_df = test_ds.to_pandas()

train_df.head()

In [ ]:
# Define binary toxicity labels
for df in [train_df, val_df, test_df]:
    df["label"] = (df["toxicity"] >= 0.5).astype(int)

train_df[["text", "toxicity", "label"]].head()

## 2.2 Establishing a Baseline Classifier

Our baseline uses:

- **TF-IDF:** Represents comments using weighted word and bigram features.
- **Logistic Regression:** Predicts whether a comment belongs to the toxic or non-toxic class.

We use `class_weight="balanced"` to account for class imbalance during training.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

# Text representation
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20_000,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
y_train = train_df["label"]
y_val = val_df["label"]

# Train classifier
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)

# Predictions on validation data
y_proba = model.predict_proba(X_val)[:, 1]
y_pred = model.predict(X_val)

print("Training completed.")
print("Number of features:", X_train.shape[1])


### 2.3 Initial Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

print (accuracy_score(y_val, y_pred))
print (f1_score(y_val, y_pred, average="macro"))

## 3. Your experiments